# Holter DL 학습

`model_choice.yaml`에서 CNN encoder, CNN+LSTM 또는 CNN+Transformer 조합을 선택합니다. `training.yaml`에서 optimizer, batch, epoch, early stopping, seed, device, workers를 설정합니다. ECG 파일 목록·환자 split·라벨은 `basic.yaml`을 사용합니다. 지도학습 코호트는 clinical, split, ECG가 모두 연결되는 환자만 포함합니다.

In [ ]:
import json
from pathlib import Path

import pandas as pd

repo = Path.cwd().resolve()
while not (repo / "Members/sang/model/DL/model_choice.yaml").is_file():
    if repo.parent == repo:
        raise RuntimeError("저장소 루트에서 노트북을 실행하세요")
    repo = repo.parent
choice_path = repo / "Members/sang/model/DL/model_choice.yaml"
from Members.sang.model.DL.training import train_holter  # noqa: E402

## 학습 실행

환자별 SQI 통과 window를 시간순으로 균등 선택하고 dataloader에서 window·lead별 z-score를 적용합니다. 기본 실행은 YAML epoch 전체를 학습합니다. 아래 실행은 실제 train cohort 전체에 적용됩니다. 짧은 실제 데이터 검증은 `max_epochs=1, limit_train_patients=16, device_override="cpu"`를 사용하세요. Test는 학습 중 사용하지 않습니다.

In [ ]:
run_dir = train_holter(choice_path)
run_dir

## Best run 확인

`DL/models/<run_id>/best_model.pt`는 validation Average Precision 기준 best epoch의 checkpoint입니다. 설정 snapshot과 환자별 validation 확률도 함께 저장됩니다. P1030은 split train이지만 clinical label이 없어 학습 대상이 아닙니다; 별도 P1030 smoke는 라벨 없이 forward만 확인합니다.

In [ ]:
print(json.loads((run_dir / "metrics.json").read_text())["validation"])
display(pd.read_csv(run_dir / "validation_predictions.csv").head())